**Steps:**
1. Divide each band into 23×23 non-overlapping quadrats
2. Compute spatial coherence γ(b) = mean normalised cross-covariance across quadrat pairs
3. Find a single optimal threshold via Rényi's entropy (α=0.5) — bands below it are noisy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score
import spectral

## Load Hyperspectral Image

In [ ]:
hdr_file_path = 'LR_234_bands.hdr'

image_data = spectral.open_image(hdr_file_path)
img = image_data.load()

original_shape = img.shape
print(f'Original Image Shape: {original_shape}')
print(f'Number of bands: {original_shape[2]}')

## Zero-Pad to Square & Transpose to (bands, rows, cols)

In [ ]:
rows, cols, n_bands = original_shape
QUADRAT_SIZE = 23
padded_dim = int(np.ceil(max(rows, cols) / QUADRAT_SIZE)) * QUADRAT_SIZE

padded_image = np.zeros((padded_dim, padded_dim, n_bands), dtype=np.float64)
padded_image[:rows, :cols, :] = img

data_cube = padded_image.transpose(2, 0, 1)
print(f'Padded shape: {padded_image.shape}, Data cube: {data_cube.shape}')

## Compute Intra-Band Spatial Coherence γ(b)

For each band, γ(b) is the mean Pearson correlation across all unique quadrat pairs.

In [ ]:
def compute_spatial_coherence(data_cube, quadrat_size=23, max_pairs=5000):
    n_bands, padded_rows, padded_cols = data_cube.shape
    nq_rows = padded_rows // quadrat_size
    nq_cols = padded_cols // quadrat_size
    n_quadrats = nq_rows * nq_cols

    print(f'Quadrats: {nq_rows}×{nq_cols} = {n_quadrats}')

    total_pairs = n_quadrats * (n_quadrats - 1) // 2
    if total_pairs > max_pairs:
        rng = np.random.default_rng(42)
        pair_set = set()
        while len(pair_set) < max_pairs:
            j, k = rng.integers(0, n_quadrats, size=2)
            if j != k:
                pair_set.add((min(j, k), max(j, k)))
        pair_indices = list(pair_set)
    else:
        pair_indices = [(j, k) for j in range(n_quadrats) for k in range(j+1, n_quadrats)]

    gamma = np.zeros(n_bands)
    for b in range(n_bands):
        band = data_cube[b]
        coh_sum, count = 0.0, 0
        for (qj, qk) in pair_indices:
            rj, cj = divmod(qj, nq_cols)
            rk, ck = divmod(qk, nq_cols)
            pj = band[rj*quadrat_size:(rj+1)*quadrat_size, cj*quadrat_size:(cj+1)*quadrat_size].flatten()
            pk = band[rk*quadrat_size:(rk+1)*quadrat_size, ck*quadrat_size:(ck+1)*quadrat_size].flatten()
            vj, vk = np.var(pj), np.var(pk)
            if vj > 0 and vk > 0:
                coh_sum += np.cov(pj, pk)[0, 1] / np.sqrt(vj * vk)
                count += 1
        gamma[b] = coh_sum / count if count > 0 else 0
        if (b+1) % 50 == 0 or b == 0:
            print(f'  Band {b+1}/{n_bands} — γ = {gamma[b]:.4f}')
    return gamma

gamma = compute_spatial_coherence(data_cube, quadrat_size=QUADRAT_SIZE)
print(f'γ range: [{gamma.min():.4f}, {gamma.max():.4f}]')

## Rényi's Entropy Thresholding (α = 0.5)

Finds a **single optimal threshold** on γ that maximises inter-class Rényi's entropy. All bands with γ ≤ threshold are flagged as noisy.

In [ ]:
def renyis_entropy_threshold(gamma, alpha=0.5):
    assert 0 < alpha < 1
    gp = np.clip(gamma, 1e-10, None)
    sorted_g = np.sort(gp)
    best_t, best_H = 0, -np.inf

    for t_idx in range(1, len(sorted_g) - 1):
        t = sorted_g[t_idx]
        c1, c2 = gp[gp <= t], gp[gp > t]
        if len(c1) > 0 and len(c2) > 0:
            s1, s2 = np.sum(c1**alpha), np.sum(c2**alpha)
            if s1 > 0 and s2 > 0:
                H = (1/(1-alpha)) * (np.log2(s1) + np.log2(s2))
                if H > best_H:
                    best_H, best_t = H, t

    return best_t, np.where(gamma <= best_t)[0]

ALPHA = 0.5
threshold, detected_noisy = renyis_entropy_threshold(gamma, alpha=ALPHA)

print(f'Single optimal threshold (α={ALPHA}): {threshold:.6f}')
print(f'Bands with γ ≤ {threshold:.6f} are noisy')
print(f'Detected noisy bands: {len(detected_noisy)}')
print(f'Indices (0-indexed): {detected_noisy}')

## Visualisation: IBSC vs Wavelength

In [ ]:
# PRISMA wavelengths: 234 bands spanning ~400–2500 nm
wavelengths = np.linspace(400, 2500, n_total)

# Try to read actual wavelengths from the header file
try:
    wl = np.array(image_data.bands.centers)
    if len(wl) == n_total:
        wavelengths = wl
        print(f'Using wavelengths from header: {wavelengths[0]:.1f} – {wavelengths[-1]:.1f} nm')
except:
    print(f'Using approximate wavelengths: {wavelengths[0]:.0f} – {wavelengths[-1]:.0f} nm')

is_noisy = np.array([i in detected_set for i in range(n_total)])

fig, ax = plt.subplots(figsize=(10, 5))

# Fill non-noisy bands (blue/purple)
gamma_clean = gamma.copy()
gamma_clean[is_noisy] = 0
ax.fill_between(wavelengths, gamma_clean, alpha=0.45, color='#7986CB', label='Non-noisy')
ax.plot(wavelengths, gamma_clean, color='#5C6BC0', linewidth=0.5)

# Fill noisy bands (green)
gamma_noisy = gamma.copy()
gamma_noisy[~is_noisy] = 0
ax.fill_between(wavelengths, gamma_noisy, alpha=0.5, color='#C5E1A5', label='Noisy')
ax.plot(wavelengths, gamma_noisy, color='#9CCC65', linewidth=0.5)

# Also plot the full continuous outline
ax.plot(wavelengths, gamma, color='#5C6BC0', linewidth=0.6, alpha=0.7)

ax.set_xlabel('Wavelength (nm)', fontsize=11)
ax.set_ylabel('IBSC', fontsize=11)
ax.set_xlim(wavelengths[0] - 10, wavelengths[-1] + 10)
ax.set_ylim(0, None)
ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
ax.tick_params(labelsize=9)

plt.tight_layout()
plt.savefig('spatial_coherence_plot.png', dpi=200, bbox_inches='tight')
plt.show()

print(f'Clean bands: {n_total - len(detected_noisy)} / {n_total}')